## Study notes for ViT
`TOPO: add related papers references by AI`

**Author: Haozhe Jia**\
This is a study note for build ViT from scratch to understand the architecture and mechanism of ViT.

In [11]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
import torch.nn as nn
import torch.nn.functional as F

def load_image(path, image_size=224):
    """Load an image file -> (1, 3, image_size, image_size) float tensor in [0, 1]."""
    img = Image.open(path).convert("RGB")
    transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
    ])
    x = transform(img).unsqueeze(0)  # add batch dim
    return x

def show_patches(x, patch_size=16):
    """BEFORE view: the original image, and the image cut into the patches
    that PatchEmbeddings' Conv2d will see."""
    img = x[0]                                      # (3, H, W)
    n = img.shape[1] // patch_size                  # patches per side, e.g. 14

    # (3, H, W) -> (3, n, n, patch_size, patch_size)
    patches = img.unfold(1, patch_size, patch_size).unfold(2, patch_size, patch_size)

    fig = plt.figure(figsize=(9, 4.5))

    ax = fig.add_subplot(1, 2, 1)
    ax.imshow(img.permute(1, 2, 0))
    ax.set_title("Original image")
    ax.axis("off")

    # right half: n x n grid of separated tiles
    for row in range(n):
        for col in range(n):
            ax = fig.add_subplot(n, 2 * n, row * 2 * n + n + col + 1)
            ax.imshow(patches[:, row, col].permute(1, 2, 0))
            ax.axis("off")

    fig.suptitle(f"{n}x{n} = {n * n} patches of {patch_size}x{patch_size} pixels")
    plt.tight_layout()
    plt.show()

def show_embeddings(x, patch_embed, use_pca=False):
    """AFTER view: run PatchEmbeddings and visualize the output sequence."""
    with torch.no_grad():
        out = patch_embed(x)                        # (1, num_patches, hidden_size)

    print(f"input : {tuple(x.shape)}")
    print(f"output: {tuple(out.shape)}  <- sequence of {out.shape[1]} tokens, "
          f"{out.shape[2]} dims each")

    tokens = out[0]                                 # (num_patches, hidden_size)
    n = int(tokens.shape[0] ** 0.5)                 # e.g. 14

    fig, axes = plt.subplots(1, 2 if use_pca else 1, figsize=(10, 4), squeeze=False)

    ax = axes[0, 0]
    im = ax.imshow(tokens, aspect="auto", cmap="viridis")
    ax.set_title("Patch embeddings (one row = one token)")
    ax.set_xlabel("hidden dim")
    ax.set_ylabel("patch index")
    fig.colorbar(im, ax=ax)

    if use_pca:
        # project 768 dims -> 3, reshape back to the 14x14 grid, show as RGB
        _, _, v = torch.pca_lowrank(tokens, q=3)
        rgb = tokens @ v                            # (num_patches, 3)
        rgb = (rgb - rgb.min(0).values) / (rgb.max(0).values - rgb.min(0).values + 1e-8)
        ax = axes[0, 1]
        ax.imshow(rgb.reshape(n, n, 3))
        ax.set_title("Same tokens, PCA to RGB on the patch grid")
        ax.axis("off")
    plt.tight_layout()
    plt.show()
    return out

# 1. PatchEmbeddings

In [ ]:
# 1. PatchEmbeddings

### In parctice

In [ ]:
class PatchEmbedding(nn.Module):
  def __init__(self, d_model, img_size, patch_size, n_channels):
    super().__init__()

    self.d_model = d_model # Dimensionality of Model
    self.img_size = img_size # Image Size
    self.patch_size = patch_size # Patch Size
    self.n_channels = n_channels # Number of Channels

    self.linear_project = nn.Conv2d(self.n_channels, self.d_model, kernel_size=self.patch_size, stride=self.patch_size)

  # B: Batch Size
  # C: Image Channels
  # H: Image Height
  # W: Image Width
  # P_col: Patch Column
  # P_row: Patch Row

  def forward(self, x):
    x = self.linear_project(x) # (B, C, H, W) -> (B, d_model, P_col, P_row)

    x = x.flatten(2) # (B, d_model, P_col, P_row) -> (B, d_model, P)

    x = x.transpose(1, 2) # (B, d_model, P) -> (B, P, d_model)

    return x


## 2. Positional Embeddings
Append class token and position tokens to the patch embeddings.
The formula of Sinusoidal Positional Embeddings:

$$
PE_{(pos,\,2k)} = \sin\!\left(\frac{pos}{10000^{2k/d}}\right), \qquad
PE_{(pos,\,2k+1)} = \cos\!\left(\frac{pos}{10000^{2k/d}}\right)
$$

The `if i \% 2 == 0}` branch handles even dimensions with $\texttt{sin}$, the `else` handles odd dimensions with $\texttt{cos}$. Each position gets a fixed vector of interleaved sines and cosines at geometrically decreasing frequencies. It's $\textbf{added}$ to the token embedding at the input:

$$
x'_{pos} = \text{embedding}_{pos} + PE_{pos}
$$

In [10]:
class PositionalEmbedding(nn.Module):
    def __init__(self, d_model, max_seq_length):
        super().__init__()
        # Create a learnable parameter named classification token
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model)) # (1, 1, d_model)

        # Creating positional encodings
        pe = torch.zeros(max_seq_length, d_model) # (max_seq_length, d_model)

        # Sinusoidal Positional Embeddings, method in paper "Attention is all you need"
        for pos in range(max_seq_length):
            for i in range(d_model):
                if i % 2 == 0:
                    pe[pos][i] = np.sin( pos / ( 10000 ** ( i / d_model )) )
                else:
                    pe[pos][i] = np.cos( pos / 10000 ** ( (i - 1) / d_model) )

        self.register_buffer('pe', pe.unsqueeze(0)) # (1, max_seq_length, d_model)

    def forward(self, x):
        # Expand to have class token for every image in batch
        # -1 in expand() means "keep this dimension as-is."
        tokens_batch = self.cls_token.expand(x.size()[0], -1, -1) # (B, 1, d_model),

        # add class tokens to the beginning of each embd
        x = torch.cat((tokens_batch, x),dim=1)

        # Adding positional encoding to embds
        x = x + self.pe

        return x

## 3. Transformers

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        # key, query, value projection/weights for all heads, but in a batch
        self.c_attn = nn.Linear(d_model, 3 * d_model)

        # output projection
        self.c_proj = nn.Linear(d_model, d_model)        # regularization
        self.n_head = n_heads
        self.d_model = d_model

    def forward(self, x):
        B, T, C = x.size() # Batch size, sequence length, embedding dimensionality (n_embd)
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.d_model, dim=2)
        k = k.view(B,T,self.n_head,C // self.n_head).transpose(1,2) # (B, nh, T, hs)
        q = q.view(B,T,self.n_head,C // self.n_head).transpose(1,2) # (B, nh, T, hs)
        v = v.view(B,T,self.n_head,C // self.n_head).transpose(1,2) # (B, nh, T, hs)

        y = F.scaled_dot_product_attention(q, k, v, is_causal=False)
        y = y.transpose(1,2).contiguous().view(B,T,C) # re-assemble all head outputs side by side
        # output projection
        y = self.c_proj(y)
        return y

class MLP(nn.Module):
    def __init__(self, d_model, r_mlp):
        super().__init__()
        self.c_fc = nn.Linear(d_model, r_mlp * d_model)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(r_mlp * d_model, d_model)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        return x

class TransformerEncoder(nn.Module):
    def __init__(self, d_model, n_heads, r_mlp=4):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads

        # Sub-Layer 1 Normalization
        self.ln_1 = nn.LayerNorm(d_model)
        # Multi-Head Attention
        self.attn = MultiHeadAttention(d_model, n_heads)
        # Sub-Layer 2 Normalization
        self.ln_2 = nn.LayerNorm(d_model)
        # MLP layer
        self.mlp = MLP(d_model,r_mlp)

    def forward(self, x):
        # Residual Connection After Sub-Layer 1
        x = x + self.attn(self.ln_1(x))

        # Residual Connection After Sub-Layer 2
        x = x + self.mlp(self.ln_2(x))

        return x